# PathSnap | Team 5 training and evaluation
UE24CS352A Machine Learning · Divyanshi (CS161), Denis Mangla (CS151), Gurubelli Yekambar Eshwar Rao (CS173), Ch Sowmya Vaishnavi (CS126)

Use a CPU runtime. Run the cells in order. Select **PathSnap_Team5_Final_Release.zip** and the organiser's **Student folder.zip** when prompted. A run uses the supplied 80 training trips, with grouped validation by trip, and the 10 student-test trips for development evaluation. Keep evaluator-reserved trips on the model laptop. Use the organiser's student data in Colab only if the organiser permits cloud processing.

This notebook starts each run from source in a clean directory and trains fresh models. Run it and save the executed notebook to show training evidence. The execution log, source and input hashes, locked package versions, charts, maps and weights are also exported. AI assistance was used to prepare the notebook and project material.

In [ ]:
import hashlib, json, os, re, shutil, stat, subprocess, sys, tempfile, time
from datetime import datetime, timezone
from pathlib import Path
from zipfile import ZipFile

is_colab = "COLAB_RELEASE_TAG" in os.environ or "google.colab" in sys.modules
if is_colab:
    from google.colab import files
    print("Select PathSnap_Team5_Final_Release.zip and Student folder.zip together.")
    uploads = files.upload()
    archives = [Path(name).resolve() for name in uploads if name.lower().endswith(".zip")]
    del uploads
else:
    archives = [Path(os.environ[k]).expanduser().resolve() for k in ("PATHSNAP_PROJECT_ZIP", "PATHSNAP_STUDENT_ZIP")]

base = Path("/content") if is_colab else Path(tempfile.gettempdir())
session = Path(tempfile.mkdtemp(prefix="pathsnap-run-", dir=base))
project, raw = session / "project", session / "student_data"
project.mkdir()
raw.mkdir()
student_trips = {"train": set(), "student_test": set()}
source_files = {}
archive_hashes = {}
for archive in archives:
    archive_hashes[archive.name] = hashlib.sha256(archive.read_bytes()).hexdigest()
    with ZipFile(archive) as bundle:
        for entry in bundle.infolist():
            parts = Path(entry.filename).parts
            if entry.is_dir() or entry.filename.startswith("/") or ".." in parts or stat.S_ISLNK(entry.external_attr >> 16):
                continue
            if len(parts) >= 3 and parts[-3] in student_trips and re.fullmatch(r"[0-9]{8}",parts[-2]):
                split, trip, name = parts[-3:]
                if name not in {trip+ext for ext in (".track", ".nodes", ".arcs", ".route")}:
                    continue
                target = raw / split / trip / name
                if target.exists():
                    raise ValueError(f"Duplicate student input: {split}/{trip}/{name}")
                student_trips[split].add(trip)
            else:
                if "project" not in parts:
                    continue
                rel = parts[parts.index("project")+1:]
                if len(rel)==1 and rel[0]=="requirements.txt":
                    pass
                elif not (len(rel)>1 and rel[0] in {"pathsnap","scripts"} and rel[-1].endswith(".py")):
                    continue
                target = project.joinpath(*rel)
                source_files[str(Path(*rel))] = hashlib.sha256(bundle.read(entry)).hexdigest()
            target.parent.mkdir(parents=True,exist_ok=True)
            with bundle.open(entry) as src, target.open("wb") as dst:
                shutil.copyfileobj(src,dst)
layout = {split:len(ids) for split,ids in student_trips.items()}
assert layout == {"train":80,"student_test":10}, f"Expected the official student partitions, got {layout}."
assert not student_trips["train"] & student_trips["student_test"], "Trip IDs overlap"
for split, ids in student_trips.items():
    for trip in ids:
        assert all((raw/split/trip/(trip+ext)).is_file() for ext in (".track",".nodes",".arcs",".route")), (split,trip)
assert all((project/p).is_file() for p in ("requirements.txt","pathsnap/run.py","scripts/compare_decoder.py")), "Use the Final Release ZIP."
assert not (project/"artifacts").exists() and not (project/"results").exists()
(project/"docs").mkdir()
os.chdir(project)
print("Clean workspace:", project)
print("Student partitions:", layout)

## 1. Install dependencies and record the run

In [ ]:
logs = project/"run_logs"
logs.mkdir(exist_ok=True)
run_record = dict(started_utc=datetime.now(timezone.utc).isoformat(), python=sys.version,
                  archive_sha256=archive_hashes, source_sha256=source_files, partitions=layout,
                  stages=[])
record_file = logs/"run_manifest.json"

def run_step(name,*args):
    command = [sys.executable,*map(str,args)]
    log = logs/f"{name}.log"
    started = time.monotonic()
    print("Running:", " ".join(command),flush=True)
    with log.open("w") as output, subprocess.Popen(command,cwd=project,text=True,bufsize=1,
                                                   stdout=subprocess.PIPE,stderr=subprocess.STDOUT) as process:
        for line in process.stdout:
            output.write(line)
            print(line,end="",flush=True)
        status = process.wait()
    run_record["stages"].append(dict(stage=name,command=command,seconds=round(time.monotonic()-started,2),exit_code=status))
    record_file.write_text(json.dumps(run_record,indent=2))
    if status:
        raise RuntimeError(f"{name} failed. See {log}")

if is_colab or os.environ.get("PATHSNAP_INSTALL","1")=="1":
    run_step("00_dependencies","-m","pip","install","-r","requirements.txt")
(logs/"versions.txt").write_text(subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, HTML

def show_png(*paths):
    for path in paths:
        display(Image(filename=str(path)))

## 2. Explore the dataset and build candidate features
Candidate coverage measures whether the aligned reference road is present in the candidate set. Model Top-1 measures whether the model selected that road.

In [ ]:
run_step("01_preprocess","-m","pathsnap.run","--data",raw,"--output","data_stage1")
ds = Path("data_stage1")
coverage = pd.read_csv(ds/"candidate_coverage.csv",dtype={"trip_id":str})
stats = coverage.groupby("split").agg(trips=("trip_id","size"),GPS_points=("observations","sum"),
    candidates=("candidate_rows","sum"),reference_hits=("candidate_hits","sum"),
    usable_alignment=("usable_alignment","sum"),ambiguous_alignment=("ambiguous_alignment","sum"))
stats["coverage_pct"]=100*stats.reference_hits/stats.GPS_points
stats.to_csv(ds/"partition_statistics.csv")
display(stats.round(3))
show_png(ds/"dataset_overview.png",ds/"training_trip_map.png")

fig, ax = plt.subplots(figsize=(9,4))
for name, group in coverage.groupby("split"):
    ax.hist(group["candidate_miss_rate"]*100,bins=np.linspace(0,100,21),alpha=.6,label=name)
ax.set(xlabel="Reference-road candidate misses (%)",ylabel="Trips",title="Candidate coverage variation by trip")
ax.legend()
fig.tight_layout()
fig.savefig(ds/"candidate_miss_distribution.png",dpi=160)
plt.show()

## 3. Train all five models
Training uses three grouped folds. Each fold holds out whole trips from the training partition. The student-test metrics use all ten development trips.

In [ ]:
run_step("02_train_five_models","-m","pathsnap.models.train","--raw",raw)
models = Path("results/models")
compare = pd.read_csv(models/"comparison.csv")
display(compare.sort_values("cv_macro_f1",ascending=False).round(4))
show_png(models/"model_comparison.png",models/"loss_runtime.png")

## 4. Per-model diagnostic plots
The confusion matrices mark the top-ranked candidate as selected once per GPS observation. ROC and precision–recall use probability scores normalized within each observation. Candidate-level F1 and mean route-class F1 answer different questions.

In [ ]:
import csv
import json
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams.update({'pdf.fonttype':42,'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False})
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import numpy as np
from sklearn.metrics import (average_precision_score, confusion_matrix, precision_recall_curve,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)

project=Path('.')
output=Path('results/model_diagnostics')
output.mkdir(parents=True,exist_ok=True)
data=project/'data_stage1'
models=project/'results/models'
splits=json.loads((data/'splits.json').read_text())['student_test_trips']
info=[
    ('random_forest','Random Forest','Divyanshi','CS161'),
    ('gaussian_nb','Gaussian Naive Bayes','Denis Mangla','CS151'),
    ('logistic_regression','Logistic Regression','Denis Mangla','CS151'),
    ('hist_gradient_boosting','HistGradientBoosting','Eshwar Rao','CS173'),
    ('knn','KNN','Sowmya Vaishnavi','CS126'),
]
blue,teal,ink,muted='#294d89','#15817c','#19263b','#516379'
trips={}
for trip in splits:
    with np.load(data/'features/student_test'/f'{trip}.npz') as f, np.load(data/'labels/student_test'/f'{trip}.npz') as l:
        trips[trip]=(f['observation_index'],f['candidate_arc_id'],l['candidate_match'].astype(bool),l['reference_arc_id'])

def evaluate(key):
    actual=[]; selected=[]; prob=[]; point_count=0; correct=0
    for trip,(obs,arc,label,truth) in trips.items():
        with np.load(models/key/'student_predictions'/f'{trip}.npz') as p:
            roads=p['predicted_arc_id']; scores=p['candidate_probability']
        predicted=arc==roads[obs]
        assert np.all(np.bincount(obs,weights=predicted,minlength=len(roads))==1)
        assert np.allclose(np.bincount(obs,weights=scores,minlength=len(roads)),1)
        assert np.array_equal(label,arc==truth[obs])
        actual.extend(label); selected.extend(predicted); prob.extend(scores)
        point_count+=len(truth);correct+=int(np.sum(roads==truth))
    tn,fp,fn,tp=confusion_matrix(actual,selected,labels=[False,True]).ravel()
    assert tp==correct and tp+fp==point_count
    precision,recall,f1,_=precision_recall_fscore_support(actual,selected,average='binary')
    details=json.loads((models/f'{key}.json').read_text())
    test=details['student_test']
    assert np.isclose(test['top1_accuracy'],correct/point_count)
    values=dict(candidate_rows=len(actual),observations=point_count,tn=tn,fp=fp,fn=fn,tp=tp,
                candidate_precision=precision,candidate_recall=recall,candidate_f1=f1,
                roc_auc=roc_auc_score(actual,prob),average_precision=average_precision_score(actual,prob),
                cv_macro_f1=details['cv_macro_f1'],student_top1=test['top1_accuracy'],
                student_macro_f1=test['macro_f1'],student_log_loss=test['log_loss'])
    return values,np.array(actual),np.array(prob),[f['metrics']['macro_f1'] for f in details['cv']]

def header(fig,eyebrow,title,sub):
    fig.text(.07,.944,eyebrow,fontsize=11,color=blue,weight='bold')
    fig.text(.07,.889,title,fontsize=25,color=ink,weight='bold')
    fig.text(.07,.847,sub,fontsize=10.5,color=muted)

records=[]
pdf=PdfPages('results/model_diagnostics/PathSnap_Model_Metrics.pdf',metadata={'Title':'PathSnap 11:30 model metrics'})
for key,name,owner,srn in info:
    v,y,p,folds=evaluate(key)
    records.append(dict(model=name,member=owner,**v))
    fig,axs=plt.subplots(2,2,figsize=(11.69,8.27))
    fig.subplots_adjust(left=.15,right=.94,top=.71,bottom=.14,hspace=.65,wspace=.40)
    header(fig,'PATHSNAP   /   11:30 METRICS   /   '+srn,name,f'{owner}   |   10 student-test trips   |   {v["observations"]:,} GPS points and {v["candidate_rows"]:,} candidate rows')
    fig.text(.07,.79,f'CV Macro F1   {v["cv_macro_f1"]:.4f}       Test Top-1   {v["student_top1"]:.2%}       Test Macro F1   {v["student_macro_f1"]:.4f}       Test log loss   {v["student_log_loss"]:.4f}',fontsize=11,weight='bold',color=ink)

    matrix=np.array([[v['tn'],v['fp']],[v['fn'],v['tp']]])
    ax=axs[0,0]
    fractions=matrix/matrix.sum(axis=1,keepdims=True)
    ax.imshow(fractions,cmap='Blues',vmin=0,vmax=1,aspect='auto')
    for (i,j),count in np.ndenumerate(matrix):
        ax.text(j,i,f'{count:,}\n{fractions[i,j]:.1%}',ha='center',va='center',
                color='white' if fractions[i,j]>.55 else ink,weight='bold',fontsize=12)
    ax.set_title('Candidate confusion matrix',loc='left',weight='bold')
    ax.set_xticks([0,1],['Rejected','Selected']);ax.set_yticks([0,1],['Other road','Reference road'])
    ax.set_xlabel('Predicted');ax.set_ylabel('Actual')
    ax.text(.5,-.36,f'Positive precision {v["candidate_precision"]:.3f}   Recall {v["candidate_recall"]:.3f}   F1 {v["candidate_f1"]:.3f}',transform=ax.transAxes,ha='center',fontsize=9,color=muted)

    ax=axs[0,1]
    fpr,tpr,_=roc_curve(y,p)
    ax.plot(fpr,tpr,color=blue,lw=2.3)
    ax.plot([0,1],[0,1],ls='--',color='#a3afc0',lw=1)
    ax.set(xlim=(0,1),ylim=(0,1.02),xlabel='False positive rate',ylabel='True positive rate')
    ax.set_title(f'Candidate ROC   |   AUC {v["roc_auc"]:.3f}',loc='left',weight='bold')
    ax.grid(alpha=.18)

    ax=axs[1,0]
    precision,recall,_=precision_recall_curve(y,p)
    ax.step(recall,precision,where='post',color=teal,lw=2.1)
    ax.axhline(y.mean(),ls='--',color='#a3afc0',lw=1)
    ax.set(xlim=(0,1),ylim=(0,1.04),xlabel='Recall',ylabel='Precision')
    ax.set_title(f'Candidate precision-recall   |   AP {v["average_precision"]:.3f}',loc='left',weight='bold')
    ax.grid(alpha=.18)

    ax=axs[1,1]
    bars=ax.bar(['Fold 0','Fold 1','Fold 2'],folds,color=blue,width=.55)
    ax.bar_label(bars,fmt='%.4f',padding=4)
    ax.axhline(v['cv_macro_f1'],ls='--',color=teal,lw=1.2)
    ax.set(ylim=(0,1),ylabel='Per-trip Macro F1')
    ax.set_title('Whole-trip cross-validation',loc='left',weight='bold')
    ax.grid(axis='y',alpha=.18)
    fig.text(.07,.071,'Confusion matrix uses one selected road per GPS point. Percentages are row-wise. ROC and PR vary the threshold over saved candidate scores.',color=muted,fontsize=10)
    fig.text(.07,.044,'Targets come from reference-route alignment. Candidate statistics and route-class Macro F1 use different units.',color=muted,fontsize=10)
    fig.savefig(output/f'{srn}_{key}.png',dpi=180);pdf.savefig(fig);plt.close(fig)

fig,axs=plt.subplots(1,2,figsize=(11.69,8.27))
fig.subplots_adjust(left=.20,right=.94,top=.72,bottom=.28,wspace=.55)
header(fig,'PATHSNAP   /   11:30 METRICS   /   TEAM SUMMARY','Five models on the same data','Random Forest led grouped validation. Logistic Regression led student Top-1. HistGradientBoosting had the lowest test log loss.')
names=['Random Forest','Gaussian NB','Logistic Regression','HistGradientBoosting','KNN']
idx=np.arange(len(names))
for ax in axs:
    ax.grid(axis='x',alpha=.18);ax.set_axisbelow(True);ax.invert_yaxis()
    ax.set_yticks(idx,names)
ax=axs[0]
for offset,k,label,color in [(-.15,'cv_macro_f1','Grouped CV Macro F1',blue),(.15,'student_macro_f1','Student Macro F1',teal)]:
    bars=ax.barh(idx+offset,[r[k] for r in records],height=.26,color=color,label=label)
    ax.bar_label(bars,fmt='%.3f',padding=4,fontsize=9)
ax.set(xlim=(0,.76),xlabel='Macro F1');ax.legend(loc='lower center',bbox_to_anchor=(.5,-.29),frameon=False,fontsize=9)
ax=axs[1]
bars=ax.barh(idx,[100*r['student_top1'] for r in records],height=.46,color=blue)
ax.bar_label(bars,fmt='%.2f%%',padding=4,fontsize=9)
ax.set(xlim=(0,100),xlabel='Student Top-1 accuracy (%)')
fig.text(.07,.119,'Show your model page first. Point to the confusion matrix, report test Top-1, then explain the grouped validation score.',fontsize=11,color=ink)
fig.text(.07,.082,'Candidate ROC AUC and average precision diagnose ranking. Grouped validation Macro F1 was used for model selection.',fontsize=10,color=muted)
fig.savefig(output/'Team_Comparison.png',dpi=180);pdf.savefig(fig);plt.close(fig);pdf.close()

with (output/'model_metrics.csv').open('w',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=list(records[0]));writer.writeheader();writer.writerows(records)
(output/'README.txt').write_text('PathSnap Team 5, 11:30 model metrics\n\nEach model uses saved predictions from the same 10 student-test trips.\nThe confusion matrix treats the selected candidate as positive and all other candidates as negative.\nROC and precision-recall use probabilities normalised within each observation.\nStudent Top-1 is GPS-point accuracy. Student Macro F1 averages road-class Macro F1 across trips.\nA reference omitted by candidate generation creates no positive candidate row and still counts as wrong in point-level Top-1.\nThe three fold scores come from whole-trip validation within training.\n')

display(pd.read_csv(output/"model_metrics.csv").round(4))
show_png(*sorted(output.glob("CS*.png")))

## 5. Tune the selected classifier and evaluate Viterbi decoding
The search starts with three random trials and then proposes seven Bayesian trials. Decoder tuning uses training validation data. Top-1, road-class Macro F1 and continuity are shown on the student-test split.

In [ ]:
run_step("03_bayesian_tuning","-m","pathsnap.models.tune","--raw",raw,"--trials",10)
Path("results/decoder").mkdir(parents=True, exist_ok=True)
run_step("04_decoder_tuning","-m","pathsnap.decoder.tune","--raw",raw)
run_step("05_decoder_evaluation","-m","pathsnap.decoder.evaluate","--raw",raw)
shutil.copyfile("results/decoder/summary.json","results/decoder/baseline_summary.json")
run_step("05a_same_road_jitter","-m","scripts.compare_decoder","--raw",raw,"--stage","jitter")
run_step("05b_transition_scale","-m","scripts.compare_decoder","--raw",raw,"--stage","scale")
run_step("05c_refined_decoder_evaluation","-m","pathsnap.decoder.evaluate","--raw",raw)
Path("results/release").mkdir(parents=True, exist_ok=True)
run_step("05d_decoder_refinement_plot","-m","scripts.plot_decoder_refinement")


tuning = json.loads(Path("results/tuning/summary.json").read_text())
decoder = json.loads(Path("results/decoder/summary.json").read_text())
display(pd.DataFrame(json.loads(Path("results/tuning/trials.json").read_text()))[["trial","method","cv_macro_f1","parameters"]])
display(pd.DataFrame({name: decoder[name] for name in ("classifier","decoder")}).loc[
    ["top1_accuracy","macro_f1","macro_precision","macro_recall","direct_continuity","inference_seconds"]].round(4))
figures = Path("results/figures")
figures.mkdir(parents=True, exist_ok=True)
trials = pd.read_json("results/tuning/trials.json").assign(display_trial=lambda d:d.trial+1)
fig, ax = plt.subplots(figsize=(10,4))
for method, series in trials.groupby("method"):
    ax.scatter(series.display_trial,series.cv_macro_f1,label=method.replace("_"," "))
ax.step(trials.display_trial,trials.cv_macro_f1.cummax(),where="post",label="Best so far")
ax.axhline(tuning["untuned_cv_macro_f1"],ls="--",color="gray",label="Baseline")
ax.set(xlabel="Trial",ylabel="Grouped CV Macro F1",xticks=trials.display_trial,title="Hyperparameter search")
ax.legend()
fig.tight_layout()
fig.savefig(figures/"Random_Forest_Tuning.png",dpi=170)
plot_stats = pd.DataFrame({name: decoder[name] for name in ("classifier","decoder")})
keys = ["top1_accuracy","macro_f1","direct_continuity"]
plot_stats.loc[keys].rename(index=dict(zip(keys,["Top-1","Trip Macro F1","Continuity"]))).plot.bar(rot=0,ylim=(0,1),title="Viterbi compared with pointwise selection",ylabel="Score",figsize=(9,4))
plt.tight_layout()
plt.savefig(figures/"Viterbi_Ablation.png",dpi=170)
trip_curves = pd.DataFrame({tag:{r["trip_id"]:r["macro_f1"] for r in decoder[path]} for tag,path in [("Pointwise","per_trip_before"),("Viterbi","per_trip_after")]})
trip_curves.plot(marker="o",ylim=(0,1),title="Trip accuracy by road-class Macro F1",ylabel="Macro F1",xlabel="Student-test trip",rot=35,figsize=(10,4))
plt.tight_layout()
plt.savefig(figures/"Per_Trip_Decoder_Results.png",dpi=170)
plt.close("all")
show_png(*sorted(figures.glob("*.png")))

## 6. Check the API and plot the road geometry
The API check replays the ten student-test trips and compares each returned road sequence with the batch decoder. Toggle road-network, reference-route, predicted-route and GPS layers in the downloaded Folium map. The reference route in this plot is for development evaluation.

In [ ]:
run_step("07_api_replay","-m","scripts.verify_release","--data",raw/"student_test")
run_step("08_geometric_evaluation","-m","scripts.release_analysis","--raw",raw)
run_step("08a_geometry_and_candidate_sweeps","-m","scripts.map_diagnostics","--raw",raw)
run_step("08b_map_review","-m","scripts.build_map_review")

display(pd.read_json("results/release/api_sweep.json"))
diag = json.loads(Path("results/release/diagnostics.json").read_text())
display(pd.DataFrame(diag["sequence"]).round(4))
display(pd.DataFrame(diag["alignment"]).round(3))
show_png("results/release/folium_layers.png","results/figures/GPS_Trajectory_and_Candidate_Zoom.png","results/figures/Dataset_Geometry_and_Sampling.png","results/figures/Candidate_Convergence.png","results/release/decoder_refinement.png")
import html
route_map = Path("results/release/PathSnap_Folium.html")
display(HTML(f'<iframe title="PathSnap road layers" width="100%" height="510" srcdoc="{html.escape(route_map.read_text(),quote=True)}"></iframe>'))

## 7. Save evidence and trained weights
Download the results ZIP and save this executed notebook with its output cells. The generated report covers the student-data experiment. For official deployment evidence, take the two screenshots during the actual laptop-to-laptop run.

In [ ]:
run_step("09_technical_report","-m","scripts.build_report")
for older_report in Path(".").glob("PathSnap_*Technical_Report.pdf"):
    shutil.move(older_report,Path("docs")/older_report.name)
run_record["finished_utc"]=datetime.now(timezone.utc).isoformat()
record_file.write_text(json.dumps(run_record,indent=2))
evidence = session/"PathSnap_Colab_Run_Evidence.zip"
with ZipFile(evidence,"w",compression=8,compresslevel=4) as bundle:
    for folder in ("artifacts","results","run_logs","docs","data_stage1"):
        for path in sorted(Path(folder).rglob("*")):
            if path.is_file() and (folder!="data_stage1" or path.parent==ds):
                bundle.write(path,path.as_posix())
with ZipFile(evidence) as bundle:
    assert bundle.testzip() is None
print(evidence, f"({evidence.stat().st_size/1e6:.1f} MB)")
if is_colab:
    files.download(str(evidence))

## Two-laptop demo
Use the trained model bundle already supplied with your final submission or the matching artifacts from a verified Colab run. Keep the weights and decoder settings together. Install dependencies on the model laptop before the review.

- **Laptop 1** serves models and data. Run `python demo.py server --data "/path/to/Student folder/student_test"` for rehearsal. For official evaluation use the reserved partition directory and add `--mode evaluator`.
- **Laptop 2** runs the UI. Run `python3 demo.py ui` from the same project folder, open `http://127.0.0.1:8090`, enter `http://LAPTOP_1_IP:8000`, and connect. Select a listed trip and click **Reconstruct route**. Both machines need a network that allows devices to reach each other.
- Capture the evaluator UI and the matching model-server POST for the same trip. Save the screenshots as `results/release/ui_demo.png` and `results/release/receiving_laptop.png`, then run `python -m scripts.build_report` from the model project.